# 08 DISK Insert

Insert DISK-derived tracking outputs into ingestion tables.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
# Setup
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

# Core imports
import datajoint as dj
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

print(f"DataJoint version: {dj.__version__}")
print(f"Database prefix: {dj.config['custom']['database.prefix']}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Load adamacs pipeline modules
from adamacs.pipeline import subject, session, equipment, surgery, event, trial, imaging, behavior, scan, model
from adamacs.schemas import mocap

print("\u2713 Pipeline modules loaded")

---
## 2. Load DISK Schema

The DISK schema is defined in `adamacs/schemas/disk.py` and provides:
- `DISKModel` - Registry of trained DISK models
- `DLCImputationTask` / `DLCImputation` - For DeepLabCut data
- `ImputationParamSet` - Refusal policy and gap ceiling, hashed
- `DLCImputation.BodyPart` / `.Gap` - Per-marker arrays and per-gap uncertainty

The motion-capture branch was removed from the schema; see `DISK_SCHEMA_DESIGN_REVIEW.md`.

In [ ]:
# Import DISK schema
from adamacs.schemas import disk

# disk.py uses deferred activation: importing it declares nothing, so bind to the
# tables that already exist on the server before using them. create_schema=False and
# create_tables=False mean this declares nothing either.
from adamacs import pipeline as _pl
_prefix = dj.config["custom"]["database.prefix"]
disk.activate(_prefix + "disk", _prefix + "pupil_tracking",
              create_schema=False, create_tables=False, linking_module=_pl)

# Show table definitions
print("DISK Schema Tables:")
print("="*50)
print(disk.DISKModel.describe())
print("\n" + "="*50)
print(disk.DLCImputationTask.describe())
print("\n" + "="*50)
print(disk.DLCImputation.describe())



In [ ]:
# Visualize schema relationships
import warnings
warnings.filterwarnings('ignore')
diagram = dj.Diagram(disk.schema) + dj.Diagram(model.PoseEstimationNew) - 1 + dj.Diagram(mocap.MotionCapture) - 1
diagram

---
## 3. Data Analysis Utilities

Functions to analyze missing data and determine if imputation is needed.

In [ ]:
def analyze_dlc_missing_data(pose_key, likelihood_threshold=0.5):
    """Analyze missing/low-confidence data in DLC pose estimation.
    
    Parameters
    ----------
    pose_key : dict
        DataJoint key for PoseEstimationNew entry
    likelihood_threshold : float
        Points below this threshold are considered missing
    
    Returns
    -------
    pd.DataFrame
        Analysis with columns: body_part, n_frames, n_nan, n_low_likelihood, pct_missing
    """
    body_parts = (model.PoseEstimationNew.BodyPartPosition & pose_key).fetch(as_dict=True)
    
    if not body_parts:
        return None
    
    results = []
    for bp in body_parts:
        x = np.array(bp['x_pos'])
        y = np.array(bp['y_pos'])
        likelihood = np.array(bp['likelihood'])
        
        n_frames = len(x)
        n_nan = np.sum(np.isnan(x) | np.isnan(y))
        n_low = np.sum(likelihood < likelihood_threshold)
        pct = 100 * (n_nan + n_low) / n_frames if n_frames > 0 else 0
        
        results.append({
            'body_part': bp['body_part'],
            'n_frames': n_frames,
            'n_nan': n_nan,
            'n_low_likelihood': n_low,
            'pct_missing': round(pct, 2)
        })
    
    return pd.DataFrame(results)

def analyze_mocap_missing_data(mocap_key):
    """Analyze missing data in motion capture tracking.
    
    Parameters
    ----------
    mocap_key : dict
        DataJoint key for MotionCapture entry
    
    Returns
    -------
    pd.DataFrame
        Analysis with columns: tracking_id, n_frames, n_nan, pct_missing
    """
    tracking = (mocap.MotionCapture.TrackingPosition & mocap_key).fetch(as_dict=True)
    
    if not tracking:
        return None
    
    results = []
    for td in tracking:
        x = np.array(td['x_pos'])
        y = np.array(td['y_pos'])
        z = np.array(td['z_pos'])
        
        n_frames = len(x)
        n_nan = np.sum(np.isnan(x) | np.isnan(y) | np.isnan(z))
        pct = 100 * n_nan / n_frames if n_frames > 0 else 0
        
        results.append({
            'tracking_id': td['tracking_id'],
            'n_frames': n_frames,
            'n_nan': n_nan,
            'pct_missing': round(pct, 2)
        })
    
    return pd.DataFrame(results)

print("\u2713 Analysis functions defined")

In [ ]:
# Example: Analyze a DLC pose estimation entry
try:
    pose_keys = model.PoseEstimationNew.fetch('KEY', limit=5)
    print(f"Found {len(pose_keys)} PoseEstimationNew entries")
    
    if pose_keys:
        df = analyze_dlc_missing_data(pose_keys[0])
        if df is not None:
            print(f"\nMissing data analysis for {pose_keys[0]}:")
            print(df.to_string(index=False))
            
            # Plot
            if df['pct_missing'].sum() > 0:
                fig, ax = plt.subplots(figsize=(10, 4))
                ax.bar(df['body_part'], df['pct_missing'])
                ax.set_ylabel('% Missing')
                ax.set_title('Missing Data by Body Part')
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                plt.show()
except Exception as e:
    print(f"No DLC data available: {e}")

---
## 4. Export Data for DISK Training

Export data from DataJoint to formats DISK can use for training.

In [ ]:
# Configuration
from pathlib import Path
import os

DISK_PATH = Path(os.environ.get('DISK_PATH', str(Path.home() / 'github' / 'DISK')))
OUTPUT_DIR = Path(os.environ.get('DISK_OUTPUT_DIR', str(Path.cwd() / 'disk_training')))

(OUTPUT_DIR / 'datasets').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'results').mkdir(parents=True, exist_ok=True)

print(f"DISK path: {DISK_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
def export_dlc_to_csv(pose_keys, output_dir, name, likelihood_threshold=0.5):
    """Export DLC data to CSV files for DISK training.
    
    Parameters
    ----------
    pose_keys : list
        List of DataJoint keys for PoseEstimationNew
    output_dir : str
        Output directory
    name : str
        Dataset name (used in filenames)
    likelihood_threshold : float
        Points below this are set to NaN
    
    Returns
    -------
    list : Paths to exported files
    """
    export_dir = os.path.join(output_dir, f"dlc_{name}")
    os.makedirs(export_dir, exist_ok=True)
    
    files = []
    for i, key in enumerate(pose_keys):
        body_parts = (model.PoseEstimationNew.BodyPartPosition & key).fetch(
            as_dict=True, order_by='body_part'
        )
        if not body_parts:
            continue
        
        n_frames = len(body_parts[0]['x_pos'])
        data = {'frame': np.arange(n_frames)}
        
        for bp in body_parts:
            kp = bp['body_part'].replace(' ', '_')
            x = np.array(bp['x_pos'], dtype=float)
            y = np.array(bp['y_pos'], dtype=float)
            lh = np.array(bp['likelihood'], dtype=float)
            
            # Set low-confidence to NaN
            x[lh < likelihood_threshold] = np.nan
            y[lh < likelihood_threshold] = np.nan
            
            data[f'{kp}_x'] = x
            data[f'{kp}_y'] = y
        
        fname = f"{name}_{i:04d}.csv"
        fpath = os.path.join(export_dir, fname)
        pd.DataFrame(data).to_csv(fpath, index=False)
        files.append(fpath)
        print(f"  Exported: {fname} ({n_frames} frames, {len(body_parts)} keypoints)")
    
    print(f"\n\u2713 Exported {len(files)} files to {export_dir}")
    return files

def export_mocap_to_npy(mocap_keys, output_dir, name):
    """Export motion capture data to NPY files for DISK training.
    
    Parameters
    ----------
    mocap_keys : list
        List of DataJoint keys for MotionCapture
    output_dir : str
        Output directory
    name : str
        Dataset name
    
    Returns
    -------
    list : Paths to exported files
    """
    export_dir = os.path.join(output_dir, f"mocap_{name}")
    os.makedirs(export_dir, exist_ok=True)
    
    files = []
    for i, key in enumerate(mocap_keys):
        tracking = (mocap.MotionCapture.TrackingPosition & key).fetch(
            as_dict=True, order_by='tracking_id'
        )
        if not tracking:
            continue
        
        n_frames = len(tracking[0]['x_pos'])
        n_markers = len(tracking)
        data = np.zeros((n_frames, n_markers, 3))
        
        for j, td in enumerate(tracking):
            data[:, j, 0] = np.array(td['x_pos'], dtype=float)
            data[:, j, 1] = np.array(td['y_pos'], dtype=float)
            data[:, j, 2] = np.array(td['z_pos'], dtype=float)
        
        fname = f"{name}_{i:04d}.npy"
        fpath = os.path.join(export_dir, fname)
        np.save(fpath, data)
        files.append(fpath)
        print(f"  Exported: {fname} ({n_frames} frames, {n_markers} markers)")
    
    # Save marker names
    if tracking:
        markers = [td['tracking_id'] for td in tracking]
        with open(os.path.join(export_dir, 'markers.txt'), 'w') as f:
            f.write('\n'.join(markers))
    
    print(f"\n\u2713 Exported {len(files)} files to {export_dir}")
    return files

print("\u2713 Export functions defined")

In [ ]:
# Example: Export DLC data (uncomment to run)
# pose_keys = model.PoseEstimationNew.fetch('KEY', limit=10)
# exported_files = export_dlc_to_csv(pose_keys, OUTPUT_DIR, 'mouse_dlc')

# Example: Export Mocap data (uncomment to run)
# mocap_keys = mocap.MotionCapture.fetch('KEY', limit=10)
# exported_files = export_mocap_to_npy(mocap_keys, OUTPUT_DIR, 'mouse_mocap')

print("Export examples ready (uncomment to run)")

---
## 5. DISK Training Commands

Generate shell commands to run DISK training pipeline.

In [ ]:
def generate_disk_commands(dataset_name, input_files, output_dir, disk_path,
                           freq=60, seq_length=60, file_type='simple_csv',
                           network='transformer', epochs=100):
    """Generate all DISK training commands.
    
    Parameters
    ----------
    dataset_name : str
        Name for the dataset
    input_files : list
        List of input file paths
    output_dir : str
        Output directory
    disk_path : str
        Path to DISK installation
    freq : int
        Data frequency in Hz
    seq_length : int
        Sequence length for training
    file_type : str
        'simple_csv' for DLC, 'npy' for mocap
    network : str
        Network type: transformer, GRU, BiGRU
    epochs : int
        Training epochs
    
    Returns
    -------
    dict : Commands for each step
    """
    files_str = ','.join(input_files)
    model_dir = f"results/{network}_{dataset_name}"
    
    commands = {
        'step1_create_dataset': f"""cd {output_dir} && python {disk_path}/DISK/create_dataset.py \\
    dataset_name={dataset_name} \\
    original_freq={freq} \\
    subsampling_freq={freq} \\
    length={seq_length} \\
    stride={seq_length // 2} \\
    file_type={file_type} \\
    dlc_likelihood_threshold=0.5 \\
    fill_gap=5 \\
    sequential=false \\
    'input_files=[{files_str}]'""",

        'step2_proba_missing': f"""cd {output_dir} && python {disk_path}/DISK/create_proba_missing_files.py \\
    dataset_name={dataset_name} \\
    indep_keypoints=True""",

        'step3_train': f"""cd {output_dir} && python {disk_path}/DISK/main_fillmissing.py \\
    network={network} \\
    hydra.run.dir={model_dir} \\
    dataset.name={dataset_name} \\
    training.epochs={epochs} \\
    training.batch_size=64 \\
    training.learning_rate=0.001 \\
    'feed_data.transforms.add_missing.files=[{dataset_name}/proba_missing.csv,{dataset_name}/proba_missing_length.csv]'""",

        'step4_test': f"""cd {output_dir} && python {disk_path}/DISK/test_fillmissing.py \\
    hydra.run.dir={model_dir}/test \\
    dataset.name={dataset_name} \\
    'evaluate.checkpoints=[{model_dir}]' \\
    evaluate.n_repeat=3""",
    }
    
    return commands, f"{output_dir}/{model_dir}"

print("\u2713 Command generator defined")

In [ ]:
# Example: Generate training commands
example_files = [f"{OUTPUT_DIR}/dlc_mouse/mouse_0000.csv", f"{OUTPUT_DIR}/dlc_mouse/mouse_0001.csv"]

commands, model_path = generate_disk_commands(
    dataset_name='mouse_dlc',
    input_files=example_files,
    output_dir=OUTPUT_DIR,
    disk_path=DISK_PATH,
    freq=60,
    seq_length=60,
    network='transformer',
    epochs=100
)

print("DISK Training Pipeline Commands")
print("=" * 60)
for step, cmd in commands.items():
    print(f"\n### {step.upper()}\n")
    print(cmd)

print(f"\n\nModel will be saved to: {model_path}")

---
## 6. Model Registration & Imputation

After training, register models and run imputation.

In [ ]:
def register_disk_model(model_name, model_path, description, data_type,
                        n_keypoints, seq_length, network_type):
    """Register a trained DISK model in the database.
    
    Parameters
    ----------
    model_name : str
        Unique identifier
    model_path : str
        Path to model directory
    description : str
        Model description
    data_type : str
        'dlc_2d', 'dlc_3d', or 'mocap_3d'
    n_keypoints : int
        Number of keypoints
    seq_length : int
        Sequence length
    network_type : str
        Network architecture
    """
    from glob import glob
    
    # Verify model exists
    if not glob(os.path.join(model_path, 'model_epoch*')):
        raise FileNotFoundError(f"No model files in {model_path}")
    
    entry = {
        'disk_model_name': model_name,
        'model_path': model_path,
        'model_description': description,
        'data_type': data_type,
        'n_keypoints': n_keypoints,
        'seq_length': seq_length,
        'network_type': network_type,
    }
    
    disk.DISKModel.insert1(entry, skip_duplicates=True)
    print(f"\u2713 Registered: {model_name}")
    return entry

# Show registered models
print("Registered DISK Models:")
print(disk.DISKModel())

In [ ]:
# Example: Register a model (uncomment after training)

# register_disk_model(
#     model_name='transformer_mouse_v1',
#     model_path=f'{OUTPUT_DIR}/results/transformer_mouse_dlc',
#     description='Transformer for mouse DLC, 9 keypoints, 60Hz',
#     data_type='dlc_2d',
#     n_keypoints=9,
#     seq_length=60,
#     network_type='transformer'
# )

print("Registration example ready (uncomment after training)")

In [ ]:
# Example: Create imputation task and run

def create_dlc_imputation_task(pose_key, disk_model_name, output_dir='', mode='trigger'):
    """Create a DLC imputation task.
    
    Parameters
    ----------
    pose_key : dict
        Key for PoseEstimationNew entry
    disk_model_name : str
        Name of registered DISK model
    output_dir : str
        Output directory for results
    mode : str
        'trigger' to run inference, 'load' to load existing
    """
    task_entry = {
        **pose_key,
        'disk_model_name': disk_model_name,
        'task_mode': mode,
        'output_dir': output_dir,
        'task_description': f'DISK imputation with {disk_model_name}',
    }
    disk.DLCImputationTask.insert1(task_entry, skip_duplicates=True)
    print(f"\u2713 Created task for {pose_key}")
    return task_entry

# Example workflow:
# 1. Get a pose estimation entry with missing data
# pose_key = (model.PoseEstimationNew).fetch('KEY', limit=1)[0]

# 2. Create imputation task
# create_dlc_imputation_task(pose_key, 'transformer_mouse_v1')

# 3. Run imputation
# disk.DLCImputation.populate(pose_key)

# 4. Query results
# imputed = (disk.DLCImputation.BodyPart & pose_key).fetch(as_dict=True)

print("Imputation workflow ready (requires trained model)")

---
## Quick Reference

### Installation
```bash
git clone https://github.com/bozeklab/DISK.git "${HOME}/github/DISK"
cd "${HOME}/github/DISK"
pip install -r DISK/requirements.txt && pip install -e .
```

### Training Pipeline
1. Export data from DataJoint: `export_dlc_to_csv()` or `export_mocap_to_npy()`
2. Generate commands: `generate_disk_commands()`
3. Run DISK training scripts in terminal
4. Register model: `register_disk_model()`
5. Create task: `create_dlc_imputation_task()`
6. Run imputation: `disk.DLCImputation.populate()`

### Key Parameters
- **seq_length**: 1 second of frames (e.g., 60 for 60Hz video)
- **stride**: seq_length/2 for 50% overlap
- **network**: 'transformer' (best), 'GRU' (faster)
- **epochs**: 50-200 depending on dataset size